# v13 — respostas aos quatro pontos bloqueantes do parecer

| # | Objecao | Onde |
|---|---|---|
| 1 | &rho; = 0,967 pode ser resposta comum a' largura de banda, nao acompanhamento; e o p-valor sobre celulas nao independentes nao vale | secao A |
| 2 | Mann-Whitney sobre 8 particoes do mesmo dado e' pseudorreplicacao — o mesmo erro que o artigo critica | secao A |
| 3 | SVD e Parkinson diferem em n, dimensionalidade e numero de folds ao mesmo tempo: a diferenca nao esta identificada | secao B |
| 4 | O piso de deteccao da SVD nunca foi medido; o limiar de 0,167 vem do sintetico | secao C |

As secoes A e B<sub>1</sub> saem de `figuras_bruto.csv`, que ja existe — segundos. As secoes B e C
exigem rodadas novas, com orcamento declarado antes de cada uma.

In [4]:
!pip install pennylane -q
import numpy as np, pandas as pd, time
import pennylane as qml
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize_scalar
from scipy.stats import spearmanr, rankdata, binomtest
from IPython.display import HTML, display
import warnings; warnings.filterwarnings("ignore")

BG, FG, OR = "#0d0d0d", "#e8e8e8", "#ff6b00"
FONT = "'JetBrains Mono','Fira Code',monospace"

# Painel escuro com acento laranja no lugar de print cru.
def painel(titulo, linhas, tabela=None, destaque=None):
    corpo = "".join(f"<div style='margin:.28em 0'><span style='color:{OR}'>$</span> {l}</div>"
                    for l in linhas)
    tab = ""
    if tabela is not None:
        tab = (tabela.style.hide(axis="index").set_table_styles([
            {"selector": "th", "props": [("color", OR), ("text-align", "right"),
                                         ("border-bottom", f"1px solid {OR}"), ("padding", "4px 12px")]},
            {"selector": "td", "props": [("text-align", "right"), ("padding", "3px 12px"),
                                         ("border-bottom", "1px solid #222")]}])
            .set_table_attributes(f"style='border-collapse:collapse;font-family:{FONT};"
                                  f"font-size:12px;color:{FG};margin:.8em 0 .2em 0'").to_html())
    dest = (f"<div style='margin-top:.9em;padding:.6em .8em;border-left:3px solid {OR};"
            f"background:#1a1206;color:{FG}'>{destaque}</div>") if destaque else ""
    display(HTML(f"<div style='background:{BG};color:{FG};font-family:{FONT};font-size:13px;"
                 f"padding:1.1em 1.3em;border:1px solid #222;border-radius:6px'>"
                 f"<div style='color:{OR};font-size:11px;letter-spacing:.16em;margin-bottom:.7em'>"
                 f"&#9632; {titulo.upper()}</div>{corpo}{tab}{dest}</div>"))

def _sf(q, reps, ent):
    dev = qml.device("default.qubit", wires=q)
    @qml.qnode(dev)
    def circ(x):
        for _ in range(reps):
            for i in range(q):
                qml.Hadamard(wires=i); qml.RZ(2.0*x[..., i], wires=i)
            if ent:
                for i in range(q-1):
                    qml.CNOT(wires=[i, i+1])
                    qml.RZ(2.0*(np.pi-x[..., i])*(np.pi-x[..., i+1]), wires=i+1)
                    qml.CNOT(wires=[i, i+1])
        return qml.state()
    return circ

def states(X, q, reps=2, ent=True, batch=512):
    f = _sf(q, reps, ent)
    return np.vstack([np.asarray(f(X[s:s+batch])) for s in range(0, len(X), batch)])

def gram_fid(A, B): return np.abs(A @ B.conj().T)**2

def fit_surrogate(K, A):
    D2 = ((A[:, None, :]-A[None, :, :])**2).sum(-1)
    r = minimize_scalar(lambda lg: np.linalg.norm(K-np.exp(-np.exp(lg)*D2)),
                        bounds=(-8, 8), method="bounded")
    return float(np.exp(r.x)), float(r.fun)/np.sqrt(K.size)

def subsample(ytr, gtr, npc, rng):
    idx = []
    for cls in (0, 1):
        take = []
        for s in rng.permutation(np.unique(gtr[ytr == cls])):
            take += list(np.where((gtr == s) & (ytr == cls))[0])
            if len(take) >= npc: break
        idx += take[:npc]
    return np.array(idx)

# A. Pontos 1 e 2 — o que a correlacao realmente mede

**Ponto 1.** Kernel quantico e surrogate caem os dois com `c`. Uma correlacao ao longo da
varredura mede, em boa parte, essa resposta comum. Aqui a correlacao bruta e' comparada com a
**correlacao parcial controlando `c`** (e depois `c` e `q`), obtida por Spearman sobre os
residuos das regressoes dos postos.

**Ponto 2.** O p-valor sai. As 48 celulas compartilham folds e nao sao independentes; o IC vem
de **bootstrap no nivel de fold**, que e' a unidade de reamostragem legitima.

Se a parcial permanecer alta, o acompanhamento e' real. Se desabar, o artigo estava reportando
a varredura.

### A0 — garantir `figuras_bruto.csv`

O arquivo e' gerado pelo notebook de figuras e vive no diretorio de trabalho do Colab, que e'
apagado quando a sessao reinicia. A celula abaixo o reconstroi se estiver ausente (&asymp; 20 min)
e o reaproveita se estiver presente (segundos). Faca download dele depois — evita repetir.

In [6]:
import os

CSV = "figuras_bruto.csv"

def carrega_svd_full(path="features_v5.pkl"):
    META = ["speaker_id", "session_id", "label", "diagnosis", "vowel", "pitch", "wav"]
    f = pd.read_pickle(path)
    F = [x for x in f.columns if x not in META]
    return f[F].values.astype(float), f.label.values.astype(int), f.speaker_id.values, "SVD"

def carrega_parkinsons(path="parkinsons.data"):
    d = pd.read_csv(path)
    subj = d.name.str.extract(r'(phon_R\d+_S\d+)')[0].values
    F = [c for c in d.columns if c not in ("name", "status")]
    return d[F].values.astype(float), d.status.values.astype(int), subj, "Parkinson"

def constroi_bruto():
    QB = [4, 6, 8, 10]
    SC = np.round(np.geomspace(0.01, 2.0, 12), 4)
    MAPAS = [("zz", True), ("prod", False)]
    partes, t0 = [], time.time()
    for fn, npc, folds in [(carrega_parkinsons, None, 4), (carrega_svd_full, 200, 10)]:
        X, y, g, nome = fn()
        rows = []
        sg = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=7)
        for fold, (tr, te) in enumerate(sg.split(X, y, g)):
            r = np.random.default_rng(fold)
            idx = subsample(y[tr], g[tr], npc, r) if npc else np.arange(len(tr))
            ytr, yte = y[tr][idx], y[te]
            imp = SimpleImputer(strategy="median").fit(X[tr][idx])
            Xtr, Xte = imp.transform(X[tr][idx]), imp.transform(X[te])
            for q in [k for k in QB if k <= Xtr.shape[1]]:
                sel = SelectKBest(f_classif, k=q).fit(Xtr, ytr)
                sc = MinMaxScaler((0, np.pi)).fit(sel.transform(Xtr))
                A0 = sc.transform(sel.transform(Xtr))
                B0 = np.clip(sc.transform(sel.transform(Xte)), 0, np.pi)
                for cv in SC:
                    A, B = A0*cv, B0*cv
                    for mp, ent in MAPAS:
                        S1, S2 = states(A, q, 2, ent), states(B, q, 2, ent)
                        K = gram_fid(S1, S1)
                        m = SVC(kernel="precomputed").fit(K, ytr)
                        gsur, _ = fit_surrogate(K, A)
                        ms = SVC(kernel="rbf", gamma=gsur).fit(A, ytr)
                        Kt = K[np.triu_indices_from(K, 1)]
                        rows.append(dict(corpus=nome, mapa=mp, fold=fold, q=q, c=cv,
                                         aq=roc_auc_score(yte, m.decision_function(gram_fid(S2, S1))),
                                         asr=roc_auc_score(yte, ms.decision_function(B)),
                                         off_sd=float(Kt.std())))
        partes.append(pd.DataFrame(rows))
        print(f"{nome} — {time.time()-t0:.0f}s")
    d = pd.concat(partes)
    d.to_csv(CSV, index=False)
    return d

if os.path.exists(CSV):
    df = pd.read_csv(CSV)
    origem = f"lido do disco ({os.path.getsize(CSV)//1024} kB)"
else:
    df = constroi_bruto()
    origem = "reconstruido nesta sessao"

zz = df[df.mapa == "zz"].copy()
painel("figuras_bruto.csv", [
    f"{len(df)} linhas &middot; {origem}",
    f"corpora: {sorted(df.corpus.unique())} &middot; mapas: {sorted(df.mapa.unique())}",
    f"qubits: {sorted(df.q.unique())} &middot; {df.c.nunique()} bandwidths"],
    destaque="Baixe este arquivo antes de fechar a sessao: todas as analises das secoes A e as "
             "figuras saem dele sem recomputar a grade.")

Parkinson — 58s
SVD — 1128s


In [7]:


def rho_parcial(d, controles):
    """Spearman entre residuos dos postos, controlando as colunas indicadas."""
    a, b = rankdata(d.aq), rankdata(d.asr)
    Z = np.column_stack([np.ones(len(d))] + [rankdata(d[k]) for k in controles])
    ra = a - Z @ np.linalg.lstsq(Z, a, rcond=None)[0]
    rb = b - Z @ np.linalg.lstsq(Z, b, rcond=None)[0]
    return float(spearmanr(ra, rb).statistic)

def boot_fold(d, fn, B=2000, seed=0):
    """IC95 por bootstrap sobre folds — a unidade legitima."""
    rng = np.random.default_rng(seed)
    folds = np.array(sorted(d.fold.unique()))
    vals = []
    for _ in range(B):
        pick = rng.choice(folds, size=len(folds), replace=True)
        sub = pd.concat([d[d.fold == f] for f in pick])
        cel = sub.groupby(["q", "c"])[["aq", "asr"]].mean().reset_index()
        try: vals.append(fn(cel))
        except Exception: pass
    v = np.array(vals)
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))

lin = []
for corpus, g in zz.groupby("corpus"):
    cel = g.groupby(["q", "c"])[["aq", "asr"]].mean().reset_index()
    bruta = float(spearmanr(cel.aq, cel.asr).statistic)
    pc = rho_parcial(cel, ["c"])
    pcq = rho_parcial(cel, ["c", "q"])
    lo, hi = boot_fold(g, lambda d: rho_parcial(d, ["c"]))
    lin.append(dict(corpus=corpus, rho_bruta=round(bruta, 3),
                    rho_parcial_c=round(pc, 3), ic95=f"[{lo:.3f}, {hi:.3f}]",
                    rho_parcial_c_q=round(pcq, 3),
                    queda=(f"{(1-abs(pc)/abs(bruta))*100:.0f}%" if abs(bruta) > 1e-9 else "—")))
res_a = pd.DataFrame(lin)

svd = res_a[res_a.corpus == "SVD"].iloc[0]
if abs(svd.rho_parcial_c) > 0.7:
    dest = ("A parcial permanece alta: o acompanhamento <b>nao</b> e' artefato da varredura. "
            "Reporte a parcial como estatistica principal e a bruta como contexto.")
elif abs(svd.rho_parcial_c) > 0.3:
    dest = ("Queda substancial. Boa parte da correlacao bruta era resposta comum a <b>c</b>. "
            "O texto precisa reportar a parcial e reformular a afirmacao central.")
else:
    dest = ("<b>A parcial desaba.</b> A correlacao bruta media a varredura, nao o "
            "acompanhamento. A afirmacao central do artigo, como esta, nao se sustenta — "
            "reescreva antes de submeter.")
painel("A1 — correlacao bruta vs parcial (ponto 1)",
       ["parcial = Spearman entre residuos dos postos, controlando c (e depois c e q)",
        "IC95 por bootstrap sobre folds, 2000 reamostras — sem p-valor (celulas nao sao independentes)"],
       tabela=res_a, destaque=dest)

corpus,rho_bruta,rho_parcial_c,ic95,rho_parcial_c_q,queda
Parkinson,0.871000,0.485000,"[0.335, 0.766]",0.354000,44%
SVD,0.973000,0.802000,"[0.229, 0.918]",0.880000,18%


### A2 — concordancia com `c` fixo

Teste complementar e mais exigente: **dentro de cada largura de banda**, os dois kernels variam
juntos entre folds? Aqui `c` esta fixo por construcao, entao nada da varredura pode contribuir.

In [8]:
lin = []
for corpus, g in zz.groupby("corpus"):
    rr = []
    for cv, gc in g.groupby("c"):
        cel = gc.groupby("fold")[["aq", "asr"]].mean()
        if len(cel) >= 4 and cel.aq.std() > 0 and cel.asr.std() > 0:
            rr.append(float(spearmanr(cel.aq, cel.asr).statistic))
    rr = np.array(rr)
    if len(rr) == 0:
        lin.append(dict(corpus=corpus, n_bandwidths=0, rho_mediano=np.nan,
                        rho_min=np.nan, rho_max=np.nan, frac_positivo="—"))
        continue
    lin.append(dict(corpus=corpus, n_bandwidths=len(rr), rho_mediano=round(np.median(rr), 3),
                    rho_min=round(rr.min(), 3), rho_max=round(rr.max(), 3),
                    frac_positivo=f"{(rr > 0).mean():.0%}"))
res_a2 = pd.DataFrame(lin)
painel("A2 — concordancia entre folds, com c fixo",
       ["um Spearman por largura de banda, sobre os folds",
        "com c fixo, a resposta comum a' varredura esta eliminada por construcao"],
       tabela=res_a2,
       destaque="Se a mediana aqui for proxima de zero enquanto a bruta e' 0,97, a correlacao "
                "global e' <b>quase toda</b> efeito de c — e o artigo precisa dizer isso. "
                "Com menos de 4 folds este teste nao roda e a linha sai vazia; nesse caso "
                "reporte apenas A1.")

corpus,n_bandwidths,rho_mediano,rho_min,rho_max,frac_positivo
Parkinson,12,1.000000,0.200000,1.000000,100%
SVD,12,0.970000,0.661000,0.988000,100%


### A3 — os pontos do Parkinson estao acima da diagonal?

O parecer observa que o deslocamento na Fig. 1 e' consistente. Teste de sinal sobre as celulas.

In [9]:
lin = []
for corpus, g in zz.groupby("corpus"):
    cel = g.groupby(["q", "c"])[["aq", "asr"]].mean().reset_index()
    d = cel.aq - cel.asr
    k = int((d > 0).sum()); n = len(d)
    p = binomtest(k, n, 0.5).pvalue
    lin.append(dict(corpus=corpus, celulas=n, acima_da_diagonal=k,
                    dif_mediana=round(d.median(), 4), p_sinal=f"{p:.2g}",
                    consistente=bool(p < 0.05)))
res_a3 = pd.DataFrame(lin)
painel("A3 — teste de sinal sobre as celulas",
       ["H0: o kernel quantico esta acima da diagonal em metade das celulas"],
       tabela=res_a3,
       destaque="Um deslocamento consistente nao e' ruido, mesmo que pequeno. Se der "
                "significativo no Parkinson, reporte junto com o piso de deteccao: "
                "<i>consistente em sinal, abaixo do piso em magnitude</i>.")

corpus,celulas,acima_da_diagonal,dif_mediana,p_sinal,consistente
Parkinson,48,34,0.021200,0.0055,True
SVD,48,22,-0.000300,0.67,False


# B. Ponto 3 — identificar o que separa os corpora

SVD e Parkinson diferem simultaneamente em **n**, **dimensionalidade** e **numero de folds**. A
escada abaixo degrada a SVD um fator por vez ate a configuracao do Parkinson. Se &rho; parcial
cair para a faixa do Parkinson, a diferenca esta identificada; se nao cair, sobra propriedade
do dominio.

In [10]:
def carrega_svd(path="features_v5.pkl"):
    META = ["speaker_id", "session_id", "label", "diagnosis", "vowel", "pitch", "wav"]
    f = pd.read_pickle(path)
    F = [x for x in f.columns if x not in META]
    return f[F].values.astype(float), f.label.values.astype(int), f.speaker_id.values

QUBITS = [4, 6, 8, 10]
SCALES = np.round(np.geomspace(0.01, 2.0, 12), 4)

def grade_rho(X, y, g, folds, npc, n_feat=None, n_amostras=None, seed=7):
    rng = np.random.default_rng(seed)
    if n_feat is not None and n_feat < X.shape[1]:
        X = X[:, rng.choice(X.shape[1], n_feat, replace=False)]
    if n_amostras is not None and n_amostras < len(y):
        falantes = rng.permutation(np.unique(g)); keep = []
        for s in falantes:
            keep += list(np.where(g == s)[0])
            if len(keep) >= n_amostras: break
        keep = np.array(keep[:n_amostras]); X, y, g = X[keep], y[keep], g[keep]
    rows = []
    sg = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=seed)
    for fold, (tr, te) in enumerate(sg.split(X, y, g)):
        r = np.random.default_rng(seed*100+fold)
        idx = subsample(y[tr], g[tr], npc, r) if npc else np.arange(len(tr))
        ytr, yte = y[tr][idx], y[te]
        imp = SimpleImputer(strategy="median").fit(X[tr][idx])
        Xtr, Xte = imp.transform(X[tr][idx]), imp.transform(X[te])
        for q in [k for k in QUBITS if k <= Xtr.shape[1]]:
            sel = SelectKBest(f_classif, k=q).fit(Xtr, ytr)
            sc = MinMaxScaler((0, np.pi)).fit(sel.transform(Xtr))
            A0 = sc.transform(sel.transform(Xtr))
            B0 = np.clip(sc.transform(sel.transform(Xte)), 0, np.pi)
            for cv in SCALES:
                A, B = A0*cv, B0*cv
                S1, S2 = states(A, q), states(B, q)
                K = gram_fid(S1, S1)
                m = SVC(kernel="precomputed").fit(K, ytr)
                gs, _ = fit_surrogate(K, A)
                ms = SVC(kernel="rbf", gamma=gs).fit(A, ytr)
                rows.append(dict(fold=fold, q=q, c=cv,
                                 aq=roc_auc_score(yte, m.decision_function(gram_fid(S2, S1))),
                                 asr=roc_auc_score(yte, ms.decision_function(B))))
    d = pd.DataFrame(rows)
    cel = d.groupby(["q", "c"])[["aq", "asr"]].mean().reset_index()
    return (float(spearmanr(cel.aq, cel.asr).statistic), rho_parcial(cel, ["c"]),
            float((cel.aq - cel.asr).abs().median()), d)

DEGRAUS = [
    ("SVD completa",                  dict(folds=10, npc=200, n_feat=None, n_amostras=None)),
    ("+ n do Parkinson (195)",        dict(folds=10, npc=60,  n_feat=None, n_amostras=195)),
    ("+ 22 features",                 dict(folds=10, npc=60,  n_feat=22,   n_amostras=195)),
    ("+ 4 folds (= Parkinson)",       dict(folds=4,  npc=60,  n_feat=22,   n_amostras=195)),
]
painel("orcamento da secao B",
       [f"{len(DEGRAUS)} degraus &times; {len(QUBITS)} qubits &times; {len(SCALES)} bandwidths",
        "o primeiro degrau e' o mais caro (SVD completa, 10 folds, 200/classe)",
        "estimativa: 25-45 min em CPU"],
       destaque="Nada e' reduzido automaticamente. Se quiser encurtar, o primeiro degrau pode "
                "ser lido de <code>figuras_bruto.csv</code> em vez de recomputado.")

In [11]:
Xs, ys, gs_ = carrega_svd()
lin, t0 = [], time.time()
for nome, cfg in DEGRAUS:
    rb, rp, dif, _ = grade_rho(Xs, ys, gs_, **cfg)
    lin.append(dict(degrau=nome, rho_bruta=round(rb, 3), rho_parcial_c=round(rp, 3),
                    dif_mediana=round(dif, 4)))
    print(f"{nome} — {time.time()-t0:.0f}s")
esc = pd.DataFrame(lin)

park = res_a[res_a.corpus == "Parkinson"].iloc[0]
alvo_b, alvo_p = float(park.rho_bruta), float(park.rho_parcial_c)
esc["dist_ao_Parkinson"] = (esc.rho_parcial_c - alvo_p).round(3)
ultimo = esc.rho_parcial_c.iloc[-1]
if abs(ultimo - alvo_p) < 0.05:
    dest = ("Igualando n, dimensionalidade e folds, a SVD <b>reproduz</b> o Parkinson. "
            "A diferenca entre corpora era desenho amostral, nao regime — corrija a secao 3.2: "
            "nao ha efeito de corpus a explicar.")
elif abs(ultimo - alvo_p) < abs(float(esc.rho_parcial_c.iloc[0]) - alvo_p)/2:
    dest = ("A escada cobre mais da metade da distancia: parte da diferenca e' desenho "
            "amostral. Reporte quanto cada fator explica e o residual como nao identificado.")
else:
    dest = ("Mesmo igualando n, features e folds, a SVD nao chega ao Parkinson. A diferenca "
            "<b>sobrevive</b> ao controle — a afirmacao condicional ao regime se sustenta.")
painel("B — escada de identificacao (ponto 3)",
       [f"alvo (Parkinson): rho bruta {alvo_b:.3f} &middot; parcial {alvo_p:.3f}",
        "cada degrau acrescenta um fator, mantendo os anteriores"],
       tabela=esc, destaque=dest)

SVD completa — 655s
+ n do Parkinson (195) — 719s
+ 22 features — 785s
+ 4 folds (= Parkinson) — 814s


degrau,rho_bruta,rho_parcial_c,dif_mediana,dist_ao_Parkinson
SVD completa,0.952000,0.802000,0.005300,0.317000
+ n do Parkinson (195),0.888000,0.436000,0.030300,-0.049000
+ 22 features,0.961000,0.863000,0.021100,0.378000
+ 4 folds (= Parkinson),0.419000,0.307000,0.035300,-0.178000


# C. Ponto 4 — piso de deteccao da SVD

O limiar de 0,167 do artigo vem do controle sintetico. Piso depende de n, balanceamento e numero
de folds — nada disso transfere. Aqui a distribuicao nula e' calculada **na propria SVD**,
permutando rotulos no nivel de falante para preservar a estrutura de grupo.

In [12]:
N_PERM = 20
Q_NULO = 8

painel("orcamento da secao C",
       [f"{N_PERM} permutacoes &times; {len(SCALES)} bandwidths &times; 10 folds, q={Q_NULO}",
        "estimativa: 40-70 min em CPU"],
       destaque="Permutacao no nivel de falante: todos os registros de um falante recebem o "
                "mesmo rotulo trocado. Permutar por gravacao destruiria a estrutura de grupo e "
                "produziria um nulo otimista.")

In [13]:
def vantagem_no_otimo(X, y, g, folds=10, npc=200, q=Q_NULO, seed=7):
    rows = []
    sg = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=seed)
    for fold, (tr, te) in enumerate(sg.split(X, y, g)):
        r = np.random.default_rng(seed*100+fold)
        idx = subsample(y[tr], g[tr], npc, r)
        ytr, yte = y[tr][idx], y[te]
        imp = SimpleImputer(strategy="median").fit(X[tr][idx])
        Xtr, Xte = imp.transform(X[tr][idx]), imp.transform(X[te])
        sel = SelectKBest(f_classif, k=q).fit(Xtr, ytr)
        sc = MinMaxScaler((0, np.pi)).fit(sel.transform(Xtr))
        A0 = sc.transform(sel.transform(Xtr))
        B0 = np.clip(sc.transform(sel.transform(Xte)), 0, np.pi)
        for cv in SCALES:
            A, B = A0*cv, B0*cv
            S1, S2 = states(A, q), states(B, q)
            K = gram_fid(S1, S1)
            m = SVC(kernel="precomputed").fit(K, ytr)
            gsur, _ = fit_surrogate(K, A)
            ms = SVC(kernel="rbf", gamma=gsur).fit(A, ytr)
            rows.append(dict(fold=fold, c=cv,
                             aq=roc_auc_score(yte, m.decision_function(gram_fid(S2, S1))),
                             asr=roc_auc_score(yte, ms.decision_function(B))))
    d = pd.DataFrame(rows)
    v = []
    for f in sorted(d.fold.unique()):
        c_star = d[d.fold != f].groupby("c").aq.mean().idxmax()
        r = d[(d.fold == f) & (d.c == c_star)].iloc[0]
        v.append(r.aq - r.asr)
    return float(np.mean(v))

def permuta_por_grupo(y, g, rng):
    grupos = np.unique(g)
    rot = {s: y[g == s][0] for s in grupos}
    novos = rng.permutation([rot[s] for s in grupos])
    saida = np.empty_like(y)
    for s, nv in zip(grupos, novos):
        saida[g == s] = nv
    return saida

obs = vantagem_no_otimo(Xs, ys, gs_)
nulo, t0 = [], time.time()
for b in range(N_PERM):
    rng = np.random.default_rng(500+b)
    nulo.append(vantagem_no_otimo(Xs, permuta_por_grupo(ys, gs_, rng), gs_))
    print(f"permutacao {b+1}/{N_PERM} — {time.time()-t0:.0f}s")
nulo = np.array(nulo)
p95 = float(np.percentile(nulo, 95))

tab = pd.DataFrame([dict(corpus="SVD", vantagem_observada=round(obs, 4),
                         nulo_mediana=round(float(np.median(nulo)), 4),
                         nulo_p95=round(p95, 4), n_perm=N_PERM,
                         acima_do_piso=bool(obs > p95))])
if obs > p95:
    dest = ("A vantagem observada supera o piso da SVD. Isso <b>contradiz</b> a conclusao atual "
            "do artigo e precisa ser investigado antes de qualquer submissao.")
else:
    dest = (f"Piso de deteccao da SVD = <b>{p95:.4f}</b> de AUC. Substitua no texto o limiar de "
            "0,167 vindo do sintetico: cada corpus passa a ter o seu, e a afirmacao de ausencia "
            "de vantagem na SVD ganha o nulo que lhe faltava.")
painel("C — distribuicao nula da SVD (ponto 4)",
       [f"{N_PERM} permutacoes de rotulo no nivel de falante, q={Q_NULO}",
        f"nulo: min {nulo.min():+.4f} &middot; mediana {np.median(nulo):+.4f} "
        f"&middot; max {nulo.max():+.4f}"],
       tabela=tab, destaque=dest)

permutacao 1/20 — 128s
permutacao 2/20 — 256s
permutacao 3/20 — 384s
permutacao 4/20 — 510s
permutacao 5/20 — 636s
permutacao 6/20 — 763s
permutacao 7/20 — 889s
permutacao 8/20 — 1016s
permutacao 9/20 — 1142s
permutacao 10/20 — 1269s
permutacao 11/20 — 1394s
permutacao 12/20 — 1521s
permutacao 13/20 — 1648s
permutacao 14/20 — 1773s
permutacao 15/20 — 1898s
permutacao 16/20 — 2025s
permutacao 17/20 — 2151s
permutacao 18/20 — 2277s
permutacao 19/20 — 2404s
permutacao 20/20 — 2529s


corpus,vantagem_observada,nulo_mediana,nulo_p95,n_perm,acima_do_piso
SVD,0.005400,0.001100,0.007500,20,False


# O que muda no manuscrito

- **Secao 3.2** passa a reportar &rho; parcial com IC bootstrap por fold; o p-valor sai, e a
  comparacao entre corpora vira faixa descritiva das 8 particoes, sem Mann-Whitney.
- **Secao 3.2** ganha o resultado da escada: quanto de n, dimensionalidade e folds explica o
  contraste, e quanto sobra.
- **Secao 3.1** passa a ter um piso por corpus. O limiar sintetico fica restrito ao controle.
- **Fig. 1** ganha o teste de sinal na legenda.